# Uji McNemar
Notebook ini menguji secara statistik apakah terdapat perbedaan performa yang signifikan antara model **MK (Monge Kantorovitch)** dan **WBC (Baseline)** menggunakan uji **McNemar's Test**.

In [67]:
!pip install statsmodels

In [68]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.contingency_tables import mcnemar
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [69]:
# Menggunakan ukuran 64x64 seperti pada training ResNet sebelumnya
data_transform = transforms.Compose([
    transforms.Resize(size=(64, 64)),
    transforms.ToTensor()
])

# Data Path (Pastikan path ini sesuai dengan letak folder data validasi)
base_mk = Path('data/kotera/Val')
base_wbc = Path('data/wbc/Val')

dataset_mk = datasets.ImageFolder(root=base_mk, transform=data_transform)
dataset_wbc = datasets.ImageFolder(root=base_wbc, transform=data_transform)

# Pastikan urutan dan jumlah data sama agar dapat di-pair untuk uji McNemar
assert len(dataset_mk) == len(dataset_wbc), 'Jumlah data tidak sama!'

loader_mk = DataLoader(dataset_mk, batch_size=128, shuffle=False)
loader_wbc = DataLoader(dataset_wbc, batch_size=128, shuffle=False)

In [70]:
# Definisi model ResNet50
class Modelku(nn.Module):
    def __init__(self, num_classes):
        super(Modelku, self).__init__()
        self.model = models.resnet50(weights='DEFAULT')
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features, num_classes)
        )
    
    def forward(self, x: torch.Tensor):
        return self.model(x)

num_classes = len(dataset_mk.classes)
model_mk = Modelku(num_classes).to(device)
model_wbc = Modelku(num_classes).to(device)

# Load Checkpoints
checkpoint_mk = 'checkpoint/resnet/bestKotera.pt'
checkpoint_wbc = 'checkpoint/resnet/bestBaseline.pt'

model_mk.load_state_dict(torch.load(checkpoint_mk, map_location=device))
model_wbc.load_state_dict(torch.load(checkpoint_wbc, map_location=device))

model_mk.eval()
model_wbc.eval()
print('Models loaded successfully.')

Models loaded successfully.


In [71]:
def evaluate_model(model, dataloader):
    correctness = []
    with torch.no_grad():
        for X, y in tqdm(dataloader, desc='Evaluasi'):
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(dim=1)
            correctness.extend((preds == y).cpu().numpy())
    return np.array(correctness)

print('Mengevaluasi model MK...')
mk_correct = evaluate_model(model_mk, loader_mk)

print('Mengevaluasi model WBC...')
wbc_correct = evaluate_model(model_wbc, loader_wbc)

Mengevaluasi model MK...


Evaluasi:   0%|          | 0/6 [00:00<?, ?it/s]

Mengevaluasi model WBC...


Evaluasi:   0%|          | 0/6 [00:00<?, ?it/s]

In [72]:
# Buat Tabel Kontingensi
# Kategori uji klasifikasi McNemar's Test:
# c00: Keduanya Model Klasifikasi Benar
# c01: MK Benar, WBC Salah
# c10: MK Salah, WBC Benar
# c11: Keduanya Model Klasifikasi Salah

c00 = np.sum((mk_correct == 1) & (wbc_correct == 1))
c01 = np.sum((mk_correct == 1) & (wbc_correct == 0))
c10 = np.sum((mk_correct == 0) & (wbc_correct == 1))
c11 = np.sum((mk_correct == 0) & (wbc_correct == 0))

table = [[c00, c01],
         [c10, c11]]

print('=== Contingency Table ===')
print(f'MK Benar & WBC Benar (c00) : {c00}')
print(f'MK Benar & WBC Salah (c01) : {c01}')
print(f'MK Salah & WBC Benar (c10) : {c10}')
print(f'MK Salah & WBC Salah (c11) : {c11}')

=== Contingency Table ===
MK Benar & WBC Benar (c00) : 610
MK Benar & WBC Salah (c01) : 67
MK Salah & WBC Benar (c10) : 66
MK Salah & WBC Salah (c11) : 7


In [73]:
# Menghitung p-value dengan uji McNemar
# Gunakan exact=True jika ada frekuensi silang < 25 (c01+c10 < 25)
exact_test = (c01 + c10) < 25
result = mcnemar(table, exact=exact_test, correction=True)

print(f'Metode Exact : {exact_test}')
print(f'Statistic    : {result.statistic:.3f}')
print(f'p-value      : {result.pvalue:.5f}')

alpha = 0.05
print('\nKesimpulan:')
if result.pvalue > alpha:
    print('Gagal menolak H0: Tidak ada perbedaan proporsi yang signifikan (performa model MK dan WBC sebanding).')
else:
    print('Menolak H0: Terdapat perbedaan proporsi yang signifikan (performa model MK dan WBC berbeda secara signifikan).')

Metode Exact : False
Statistic    : 0.000
p-value      : 1.00000

Kesimpulan:
Gagal menolak H0: Tidak ada perbedaan proporsi yang signifikan (performa model MK dan WBC sebanding).
